# 中老年人群高血脂症风险预警及干预优化

本 Notebook 从原始附件直接读取数据，依次完成问题一、问题二和问题三。所有患者参数、阈值、约束检查和论文引用数值均可追溯到 `code/outputs/`。


In [1]:
from code.solution import (
    PARAMS, OUTPUT_DIR, build_run_summary, configure_plotting, load_and_preprocess,
    run_problem1, run_problem2, run_problem3, write_json, write_results_report
)
import time
start_time = time.perf_counter()
font_name = configure_plotting()
print(f"绘图字体：{font_name}")


绘图字体：Microsoft YaHei


In [2]:
df, data_quality = load_and_preprocess()
print(data_quality)


数据读取完成：1000 行，痰湿质患者 278 人
诊断标签与四项血脂异常规则一致率：100.0%
{'rows': 1000, 'columns': 44, 'duplicate_sample_ids': 0, 'missing_values': 0, 'label_prevalence': 0.793, 'diagnostic_rule_agreement': 1.0, 'phlegm_constitution_count': 278, 'activity_identity_max_error': 0.0}


In [3]:
res1 = run_problem1(df, PARAMS)
print(res1["screen"].round(4).to_string(index=False))
print(res1["constitution"].round(4).to_string(index=False))


问题1：筛选出 3 个双终点稳定指标：['甘油三酯_TG', '总胆固醇_TC', '血尿酸']
九体质调整 OR 排名首位为 气虚质积分：OR=1.127，95%CI [0.961, 1.322]，P=0.141
        指标名称  痰湿终点入选频率  血脂终点入选频率  痰湿平均重要性  血脂平均重要性   共识得分  痰湿Spearman_rho  痰湿相关P值  单指标诊断AUC 风险方向
     甘油三酯_TG      0.98      1.00   0.0790   0.3688 0.8590         -0.0158  0.6186    0.8482   正向
     总胆固醇_TC      0.98      1.00   0.0842   0.3357 0.8464         -0.0352  0.2659    0.8103   正向
         血尿酸      0.98      1.00   0.0771   0.1567 0.5532         -0.0096  0.7612    0.6455   正向
低密度脂蛋白_LDL_C      0.96      1.00   0.0783   0.0624 0.3482          0.0099  0.7551    0.6310   正向
高密度脂蛋白_HDL_C      1.00      1.00   0.0830   0.0407 0.2956          0.0139  0.6607    0.6180   负向
         BMI      1.00      0.62   0.1049   0.0047 0.0892         -0.0220  0.4866    0.5189   正向
          血糖      0.98      0.66   0.0831   0.0054 0.0868         -0.0205  0.5181    0.5285   负向
      活动量表总分      0.76      0.36   0.0612   0.0036 0.0396         -0.0178  0.5734    0.5145   正向
 体质类型    回归系数  调整OR

In [4]:
res2 = run_problem2(df, PARAMS)
print(res2["candidate_models"].round(4).to_string(index=False))
print(res2["test_tiers"].round(4).to_string(index=False))
print(res2["combinations"].head(10).round(4).to_string(index=False))


问题2：选用 随机森林，独立测试 AUC=0.821，Brier=0.135
训练集 OOF 阈值：低/中=0.5241，中/高=0.9167
测试集低/中/高人数：[43, 117, 140]；实际患病率：[0.605, 0.632, 0.986]
痰湿体质高危核心组合首位：痰湿体质 + 痰湿积分≥60 + 尿酸异常
        模型  训练集CV_AUC均值  训练集CV_AUC标准差
      随机森林       0.8653        0.0302
加权Logistic       0.6350        0.0784
  数据集  风险等级编码 风险等级  人数     占比  实际患病率  患病率95%CI下限  患病率95%CI上限
独立测试集       1   低危  43 0.1433 0.6047      0.4556      0.7401
独立测试集       2   中危 117 0.3900 0.6325      0.5427      0.7157
独立测试集       3   高危 140 0.4667 0.9857      0.9550      0.9970
                 核心特征组合  覆盖人数  总体支持度  高危组支持度  高危置信度    提升度
  痰湿体质 + 痰湿积分≥60 + 尿酸异常    27 0.0900  0.1714 0.8889 1.9048
            痰湿体质 + 尿酸异常    55 0.1833  0.3143 0.8000 1.7143
      痰湿体质 + 尿酸异常 + 吸烟史    32 0.1067  0.1786 0.7812 1.6741
痰湿体质 + 痰湿积分≥60 + 血糖>6.1     9 0.0300  0.0500 0.7778 1.6667
   痰湿体质 + 活动总分<40 + 吸烟史     9 0.0300  0.0500 0.7778 1.6667
   痰湿体质 + 血糖>6.1 + 尿酸异常     9 0.0300  0.0500 0.7778 1.6667
      痰湿体质 + 尿酸异常 + 饮酒史    32 0.1067  0.1714 0.7500 1.6071
         

In [5]:
res3 = run_problem3(df, PARAMS)
print(res3["summary"][res3["summary"]["样本ID"].isin([1,2,3])].round(4).to_string(index=False))
print(res3["plans_123"].round(4).to_string(index=False))
print(res3["matching"].round(4).to_string(index=False))


问题3：已对全部 278 名痰湿质患者求解，约束回代全部通过=True
样本1：64.0→48.5分，成本1014元，最大允许强度1级
样本2：58.0→37.0分，成本1240元，最大允许强度2级
样本3：59.0→33.0分，成本1674元，最大允许强度3级
 样本ID  年龄组  活动量表总分  初始痰湿积分      初始调理等级  最大允许活动强度  推荐首月活动强度  推荐首月每周次数  最终痰湿积分  降低分数   降低比例  最大可降低分数  效果保留率  六个月总成本
    1    2    38.0    64.0   强化调理(≥62)         1         1        10    48.5  15.5 0.2422     17.0 0.9118    1014
    2    1    40.0    58.0   基础调理(≤58)         2         2        10    37.0  21.0 0.3621     23.0 0.9130    1240
    3    1    63.0    59.0 中度调理(59-61)         3         3        10    33.0  26.0 0.4407     28.5 0.9123    1674
 样本ID  月份  月初痰湿积分  调理等级              核心调理方式  活动强度  单次时长_分钟  每周次数  当月调理成本  当月活动成本  当月总成本  当月活动降幅  月末痰湿积分
    1   1    64.0     3 饮食调理+穴位按摩+八段锦+中药代茶饮     1       10    10     130     120    250    0.05    61.0
    1   2    61.0     2       饮食调理+穴位按摩+八段锦     1       10    10      80     120    200    0.05    58.0
    1   3    58.0     1           饮食调理+穴位按摩     1       10    10      30     120    150    0.05    5

In [6]:
elapsed_seconds = time.perf_counter() - start_time
write_results_report(data_quality, res1, res2, res3, elapsed_seconds)
run_summary = build_run_summary(font_name, data_quality, res1, res2, res3, elapsed_seconds)
write_json(OUTPUT_DIR / "run_summary.json", run_summary)
print(f"完整流程通过，耗时 {elapsed_seconds:.1f} 秒。")
print("所有 CSV/JSON 已保存到 code/outputs，PDF 图已保存到 figures。")


完整流程通过，耗时 89.0 秒。
所有 CSV/JSON 已保存到 code/outputs，PDF 图已保存到 figures。
